<a href="https://colab.research.google.com/github/jsmak/mytransformer/blob/main/notebooks/11_Transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Transformer**
지금까지 구현해 온 각 클래스를 사용해, 「Transformer」를 구현합니다.

#### **(1) ScaledDotProductAttention 클래스**
「Scaled Dot-Product Attention」의 클래스입니다.

In [21]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_head):
        super().__init__()
        self.d_head = d_head

    def forward(self, Q, K, V, attn_mask=None):
        attention_scores = Q @ K.transpose(-2, -1)/ np.sqrt(self.d_head)

        if attn_mask is not None:
            attention_scores = attention_scores.masked_fill(attn_mask == 0, float("-inf"))

        attention_weights = F.softmax(attention_scores, dim=-1)
        context_vector = attention_weights @ V
        return context_vector

#### **(2) MultiHeadAttention 클래스**
Multi-Head Attention」클래스입니다.

In [22]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.out = nn.Linear(d_model, d_model, bias=False)

        self.attention = ScaledDotProductAttention(self.d_head)

    def forward(self, xq, xk, xv, attn_mask=None):
        batch_size = xq.size(0)
        Q = self.W_q(xq).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        K = self.W_k(xk).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        V = self.W_v(xv).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)

        # Apply Scaled Dot Product Attention
        x = self.attention(Q, K, V, attn_mask)

        # Concatenate and apply final linear
        head_output = x.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        out = self.out(head_output)
        return out

#### **(3) Attention mask 생성**
Attention mask를 생성합니다..

In [23]:
def make_attention_mask(pad_mask=None, q_len=None, causal=False):
    attn_mask = None

    if pad_mask is not None:
        batch_size, k_len = pad_mask.shape

        if q_len is None:
            q_len = k_len
        attn_mask = pad_mask.unsqueeze(1).unsqueeze(2)
        attn_mask = attn_mask.expand(batch_size, 1, q_len, k_len)

    if causal:
        if q_len is None:
            raise ValueError("causal=True일 때는 q_len이 필요합니다.")

        k_len = q_len
        causal_mask = torch.tril(torch.ones(q_len, k_len))

        # [1, 1, q_len, k_len]
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)

        if attn_mask is None:
            attn_mask = causal_mask
        else:
            attn_mask = attn_mask * causal_mask
    return attn_mask

**(4) EncoderLayer 클래스**  
「Attention Is All You Need」논문의, 1회 Encoder동작을 처리를 수행하는 EncoderLayer 클래스입니다.

In [24]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.multihead_attn = MultiHeadAttention(d_model, num_heads)

        d_ff = d_model * 4

        self.W_ffn1 = nn.Parameter(torch.randn(d_model, d_ff))
        self.b_ffn1 = nn.Parameter(torch.randn(d_ff))
        self.W_ffn2 = nn.Parameter(torch.randn(d_ff, d_model))
        self.b_ffn2 = nn.Parameter(torch.randn(d_model))

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, pad_mask=None):
        # Encoder Self-Attention: Q = K = V = x
        context_vector = self.multihead_attn(x, x, x, pad_mask)

        norm1_out = self.norm1(x + context_vector)

        f_out = F.relu(norm1_out @ self.W_ffn1 + self.b_ffn1)
        f_out = f_out @ self.W_ffn2 + self.b_ffn2

        final_out = self.norm2(norm1_out + f_out)
        return final_out

**(5) Encoder 클래스**  
「Attention Is All You Need」 논문의 EncoderLayer 클래스를 반복하는 Encoder 클래스입니다.

In [25]:
class Encoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer(d_model, num_heads) for _ in range(num_layers)])

    def forward(self, x, pad_mask=None):
        src_len = x.size(1)
        self_mask = make_attention_mask(pad_mask=pad_mask, q_len=src_len, causal=False)
        for layer in self.layers:
            x = layer(x, None)
        return x

**(6) DecoderLayer 클래스**  
「Attention Is All You Need」논문의, 1회 Decoder동작을 처리를 수행하는 DecoderLayer 클래스입니다.

In [26]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()

        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)

        d_ff = d_model * 4

        self.W_ffn1 = nn.Parameter(torch.randn(d_model, d_ff))
        self.b_ffn1 = nn.Parameter(torch.randn(d_ff))
        self.W_ffn2 = nn.Parameter(torch.randn(d_ff, d_model))
        self.b_ffn2 = nn.Parameter(torch.randn(d_model))

        # Decoder는 Add & Norm이 3개
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

    def forward(self, x, encoder_out, tgt_self_mask=None, cross_mask=None):
        # 1. Masked Multi-Head Self-Attention
        self_attn_out = self.self_attn(x, x, x, attn_mask=tgt_self_mask)
        norm1_out = self.norm1(x + self_attn_out)

        # 2. Encoder-Decoder Attention, Cross Attention
        cross_attn_out = self.cross_attn(norm1_out, encoder_out, encoder_out, attn_mask=cross_mask)
        norm2_out = self.norm2(norm1_out + cross_attn_out)

        # 3. Feed Forward Network
        f_out = F.relu(norm2_out @ self.W_ffn1 + self.b_ffn1)
        f_out = f_out @ self.W_ffn2 + self.b_ffn2

        # 4. Add & Norm
        final_out = self.norm3(norm2_out + f_out)
        return final_out

**(7) Decoder 클래스**  
「Attention Is All You Need」 논문의 DecoderLayer 클래스를 반복하는 Encoder 클래스입니다.

In [27]:
class Decoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads):
        super().__init__()

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, num_heads)
            for _ in range(num_layers)
        ])

    def forward(self, x, encoder_out, tgt_pad_mask=None, src_pad_mask=None):
        tgt_len = x.size(1)
        tgt_self_mask = make_attention_mask(pad_mask=tgt_pad_mask, q_len=tgt_len, causal=True)
        cross_mask = make_attention_mask(pad_mask=src_pad_mask, q_len=tgt_len, causal=False )

        for layer in self.layers:
            x = layer(x, encoder_out, tgt_self_mask=tgt_self_mask, cross_mask=cross_mask)

        # padding 위치 출력을 0으로 유지하고 싶을 경우
        if tgt_pad_mask is not None:
            x = x * tgt_pad_mask.unsqueeze(-1)
        return x

**(8) PositionalEncoding 클래스**  
「Attention Is All You Need」 논문의 Position Encoding 클래스입니다.

In [28]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        self.pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)

        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(torch.log(torch.tensor(10000.0)) / d_model))

        self.pe[:, 0::2] = torch.sin(position * div_term)
        self.pe[:, 1::2] = torch.cos(position * div_term)
        self.pe = self.pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

**(9) Transformer 클래스**  
「Attention Is All You Need」 논문의 Transformer를 클래스로 구현합니다.

In [29]:
class Transformer(nn.Module):
    def __init__(self, d_model=512, nhead=8, num_encoder_layers=6, num_decoder_layers=6, vocab_size=None):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.encoder = Encoder(num_encoder_layers, d_model, nhead)
        self.decoder = Decoder(num_decoder_layers, d_model, nhead)
        self.fc_out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, src_pad_mask=None, tgt_pad_mask=None):
        src_emb = self.pos_encoder(self.embedding(src) * math.sqrt(self.d_model))
        tgt_emb = self.pos_encoder(self.embedding(tgt) * math.sqrt(self.d_model))
        memory = self.encoder(x=src_emb, pad_mask=src_pad_mask)
        output = self.decoder(x=tgt_emb, encoder_out=memory, tgt_pad_mask=tgt_pad_mask, src_pad_mask=src_pad_mask)
        logits = self.fc_out(output)
        return logits

**(10)  Transformer 클래스 실행**   
Transformer 클래스를 실행합니다.

In [30]:
# 파라미터 설정
d_model = 512  # 입력 차원의 수
n_head = 8     # 어텐션 헤드 수
num_layers = 6 # 인코더 레이어 수
batch_size = 32 # 배치 크기

num_encoder_layers = 6
num_decoder_layers = 6

vocab_size = 10000

tgt_seq_len = 100   # 디코더 입력 시퀀스의 길이
src_seq_len = 120   # 인코더 시퀀스 길이

# 인코더 인스턴스화
transformer_model = Transformer(d_model, n_head, num_encoder_layers, num_decoder_layers, vocab_size=vocab_size)

# 모델을 평가 모드로 설정
transformer_model.eval()

# 더미 입력 데이터
decode_input_tensor = torch.rand(batch_size, tgt_seq_len, d_model) # 디코더 입력 텐서
enc_output = torch.rand(batch_size, src_seq_len, d_model)   # 인코더의 출력 텐서

src = torch.randint(0, vocab_size, (batch_size, src_seq_len))
tgt = torch.randint(0, vocab_size, (batch_size, tgt_seq_len))

src_mask = torch.ones(batch_size, src_seq_len)
tgt_mask = torch.ones(batch_size, tgt_seq_len)

# 트랜스포머에 데이터를 전달
with torch.no_grad(): # 기울기 계산하지 않음
  output = transformer_model(src, tgt, src_mask, tgt_mask)

print("Transformer input shape:", decode_input_tensor.shape )   # 입력 텐서의 shape: [batch_size, seq_len, d_model]
print("Transformer output shape:", output.shape) # 출력 텐서의 shape:  [batch_size, tgt_seq_len, tgt_vocab_size]
print("Transformer output:", output)

Transformer input shape: torch.Size([32, 100, 512])
Transformer output shape: torch.Size([32, 100, 10000])
Transformer output: tensor([[[-0.4352,  0.6610, -0.0260,  ...,  1.4471,  0.6506, -0.1951],
         [-0.1961,  0.8289, -0.1252,  ...,  1.0004,  0.5345, -0.5307],
         [-0.0492,  0.6968, -0.3646,  ...,  1.1760,  0.6074, -0.2325],
         ...,
         [ 0.3232,  0.4751, -0.1642,  ...,  1.0594,  0.1463, -0.7899],
         [-0.0419,  0.6937, -0.4024,  ...,  0.7937,  0.1870, -0.5302],
         [-0.4101,  0.4811,  0.0335,  ...,  1.1162,  0.4093, -0.5750]],

        [[ 0.2916,  0.4171, -0.5166,  ...,  0.5606,  0.1281, -0.3952],
         [ 0.3075,  0.4760, -0.1481,  ...,  0.5675,  0.5213, -0.4369],
         [-0.1334,  0.2998, -0.3923,  ...,  0.4651,  0.3244, -0.6366],
         ...,
         [-0.2889,  0.4234, -0.3016,  ...,  0.9261,  0.8186, -0.2236],
         [-0.4404,  0.1614, -0.1873,  ...,  0.8260,  0.5213, -0.6063],
         [-0.3161,  0.3222, -0.1605,  ...,  0.6817,  0.1665, -